# 使用 Milvus 和 OpenAI 进行过滤搜索
### 寻找你的下一部电影

在本 notebook 中，我们将使用 OpenAI 为电影简介生成嵌入，并在 Milvus 中利用这些嵌入查找相关电影。为了缩小搜索结果范围并尝试一些新东西，我们将使用过滤来进行元数据搜索。本示例的数据集来自 HuggingFace datasets，包含八千多条电影记录。

首先下载本 notebook 所需的库：
- `openai` 用于与 OpenAI 嵌入服务通信
- `pymilvus` 用于与 Milvus 服务器通信
- `datasets` 用于下载数据集
- `tqdm` 用于显示进度条


In [ ]:
! pip install openai pymilvus datasets tqdm

安装好所需的包后就可以开始了。首先启动 Milvus 服务。运行的文件是本文件所在文件夹中的 `docker-compose.yaml`。该命令会启动一个 Milvus 单机实例，供本次测试使用。  

In [4]:
! docker compose up -d

E0317 14:06:38.344884000 140704629352640 fork_posix.cc:76]             Other threads are currently calling into gRPC, skipping fork() handlers


[+] Running 1/0
 ⠿ Network milvus          Created                                         0.1s
 ⠋ Container milvus-etcd   Creating                                        0.0s
 ⠋ Container milvus-minio  Creating                                        0.0s
[+] Running 1/3
 ⠿ Network milvus          Created                                         0.1s
 ⠙ Container milvus-etcd   Creating                                        0.1s
 ⠙ Container milvus-minio  Creating                                        0.1s
[+] Running 2/3
 ⠿ Network milvus               Created                                    0.1s
 ⠿ Container milvus-etcd        Starting                                   0.2s
 ⠿ Container milvus-minio       Starting                                   0.2s
 ⠿ Container milvus-standalone  Created                                    0.1s
[+] Running 2/4
 ⠿ Network milvus               Created                                    0.1s
 ⠿ Container milvus-etcd        Starting                

Milvus 运行起来后，我们可以设置全局变量：
- HOST：Milvus 主机地址
- PORT：Milvus 端口号
- COLLECTION_NAME：在 Milvus 中为集合取的名称
- DIMENSION：嵌入的维度
- OPENAI_ENGINE：要使用的嵌入模型
- openai.api_key：你的 OpenAI 账户密钥
- INDEX_PARAM：集合使用的索引设置
- QUERY_PARAM：要使用的搜索参数
- BATCH_SIZE：一次嵌入并插入多少部电影

In [30]:
import openai

HOST = 'localhost'
PORT = 19530
COLLECTION_NAME = 'movie_search'
DIMENSION = 1536
OPENAI_ENGINE = 'text-embedding-3-small'
openai.api_key = 'sk-your_key'

INDEX_PARAM = {
    'metric_type':'L2',
    'index_type':"HNSW",
    'params':{'M': 8, 'efConstruction': 64}
}

QUERY_PARAM = {
    "metric_type": "L2",
    "params": {"ef": 64},
}

BATCH_SIZE = 1000

In [6]:
from pymilvus import connections, utility, FieldSchema, Collection, CollectionSchema, DataType

# Connect to Milvus Database
connections.connect(host=HOST, port=PORT)

In [18]:
# Remove collection if it already exists
if utility.has_collection(COLLECTION_NAME):
    utility.drop_collection(COLLECTION_NAME)

In [19]:
# Create collection which includes the id, title, and embedding.
fields = [
    FieldSchema(name='id', dtype=DataType.INT64, is_primary=True, auto_id=True),
    FieldSchema(name='title', dtype=DataType.VARCHAR, max_length=64000),
    FieldSchema(name='type', dtype=DataType.VARCHAR, max_length=64000),
    FieldSchema(name='release_year', dtype=DataType.INT64),
    FieldSchema(name='rating', dtype=DataType.VARCHAR, max_length=64000),
    FieldSchema(name='description', dtype=DataType.VARCHAR, max_length=64000),
    FieldSchema(name='embedding', dtype=DataType.FLOAT_VECTOR, dim=DIMENSION)
]
schema = CollectionSchema(fields=fields)
collection = Collection(name=COLLECTION_NAME, schema=schema)

In [20]:
# Create the index on the collection and load it.
collection.create_index(field_name="embedding", index_params=INDEX_PARAM)
collection.load()

## 数据集
Milvus 启动并运行后，就可以开始获取数据了。Hugging Face Datasets 是一个汇集了大量用户数据集的平台，本示例使用的是 HuggingLearners 的 netflix-shows 数据集。该数据集包含八千多部电影及其元数据。我们将嵌入每条简介，并把它与标题、类型、上映年份和分级一起存入 Milvus。

In [21]:
import datasets

# Download the dataset 
dataset = datasets.load_dataset('hugginglearners/netflix-shows', split='train')

Found cached dataset csv (/Users/filiphaltmayer/.cache/huggingface/datasets/hugginglearners___csv/hugginglearners--netflix-shows-03475319fc65a05a/0.0.0/6b34fb8fcf56f7c8ba51dc895bfa2bfbe43546f190a60fcf74bb5e8afdcc2317)


## 插入数据
数据已经下载到本地，现在可以开始对其进行嵌入并插入 Milvus。嵌入函数接收文本，并以列表形式返回嵌入。 

In [22]:
# Simple function that converts the texts to embeddings
def embed(texts):
    embeddings = openai.Embedding.create(
        input=texts,
        engine=OPENAI_ENGINE
    )
    return [x['embedding'] for x in embeddings['data']]


下一步执行实际的插入操作。我们遍历所有条目并组成批次，每当达到设定的批大小时就插入一次。循环结束后，如果还剩最后一批，则将其插入。 

In [26]:
from tqdm import tqdm

data = [
    [], # title
    [], # type
    [], # release_year
    [], # rating
    [], # description
]

# Embed and insert in batches
for i in tqdm(range(0, len(dataset))):
    data[0].append(dataset[i]['title'] or '')
    data[1].append(dataset[i]['type'] or '')
    data[2].append(dataset[i]['release_year'] or -1)
    data[3].append(dataset[i]['rating'] or '')
    data[4].append(dataset[i]['description'] or '')
    if len(data[0]) % BATCH_SIZE == 0:
        data.append(embed(data[4]))
        collection.insert(data)
        data = [[],[],[],[],[]]

# Embed and insert the remainder 
if len(data[0]) != 0:
    data.append(embed(data[4]))
    collection.insert(data)
    data = [[],[],[],[],[]]


100%|██████████| 8807/8807 [00:31<00:00, 276.82it/s]


## 查询数据库
数据已安全地插入 Milvus，现在可以执行查询了。查询接收一个元组，包含你要搜索的电影描述以及要使用的过滤条件。有关过滤条件的更多信息，请参阅[这里](https://milvus.io/docs/boolean.md)。搜索会先打印出你的描述和过滤表达式，然后针对每个结果打印结果电影的分数、标题、类型、上映年份、分级和简介。

In [33]:
import textwrap

def query(query, top_k = 5):
    text, expr = query
    res = collection.search(embed(text), anns_field='embedding', expr = expr, param=QUERY_PARAM, limit = top_k, output_fields=['title', 'type', 'release_year', 'rating', 'description'])
    for i, hit in enumerate(res):
        print('Description:', text, 'Expression:', expr)
        print('Results:')
        for ii, hits in enumerate(hit):
            print('\t' + 'Rank:', ii + 1, 'Score:', hits.score, 'Title:', hits.entity.get('title'))
            print('\t\t' + 'Type:', hits.entity.get('type'), 'Release Year:', hits.entity.get('release_year'), 'Rating:', hits.entity.get('rating'))
            print(textwrap.fill(hits.entity.get('description'), 88))
            print()

my_query = ('movie about a fluffly animal', 'release_year < 2019 and rating like \"PG%\"')

query(my_query)

Description: movie about a fluffly animal Expression: release_year < 2019 and rating like "PG%"
Results:
	Rank: 1 Score: 0.30083978176116943 Title: The Lamb
		Type: Movie Release Year: 2017 Rating: PG
A big-dreaming donkey escapes his menial existence and befriends some free-spirited
animal pals in this imaginative retelling of the Nativity Story.

	Rank: 2 Score: 0.33528298139572144 Title: Puss in Boots
		Type: Movie Release Year: 2011 Rating: PG
The fabled feline heads to the Land of Giants with friends Humpty Dumpty and Kitty
Softpaws on a quest to nab its greatest treasure: the Golden Goose.

	Rank: 3 Score: 0.33528298139572144 Title: Puss in Boots
		Type: Movie Release Year: 2011 Rating: PG
The fabled feline heads to the Land of Giants with friends Humpty Dumpty and Kitty
Softpaws on a quest to nab its greatest treasure: the Golden Goose.

	Rank: 4 Score: 0.3414868116378784 Title: Show Dogs
		Type: Movie Release Year: 2018 Rating: PG
A rough and tough police dog must go undercover